In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import numpy as np
import os
import time

In [ ]:
# Constants
RANDOMSTATE = 1082
MODEL_NAME = "Logistic Regression (SMOTE)"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.smote_heldout("uav")
X_train_resampled = EXP.X_train
y_train_resampled = EXP.y_train
X_test_scaled = EXP.X_test
y_test = EXP.y_test


In [5]:
# Train the model
model = LogisticRegression(
    random_state=RANDOMSTATE,
    max_iter=2000,
    solver='saga'
)

train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [6]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 77.85%
F1 Score: 0.8013

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.81      0.86      2500
           1       0.32      0.56      0.41       398

    accuracy                           0.78      2898
   macro avg       0.62      0.69      0.64      2898
weighted avg       0.84      0.78      0.80      2898



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="SMOTE",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
